# imports and setup

In [ ]:
import joblib
import re
import string
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Load the saved assets
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('vectorizer.pkl')

print("✅ Model, Vectorizer, and API Client Ready!")

# Helper Functions (Preprocessing)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\@\w+|\#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    return text.strip()

def get_ai_response(user_text):
    prompt = f"Respond with empathy and supportive guidance to: {user_text}"
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return completion.choices[0].message.content

# Interactive Test Loop

In [ ]:
def test_chatbot():
    print("--- AI Mental Health Chatbot Test (Type 'quit' to exit) ---")
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() == 'quit': break
        
        # Clean & Vectorize
        cleaned = clean_text(user_input)
        vec_text = vectorizer.transform([cleaned])
        
        # Predict Sentiment
        sentiment = model.predict(vec_text)[0]
        print(f"DEBUG: Predicted Sentiment -> {sentiment}")
        
        # Get empathetic response
        response = get_ai_response(user_input)
        print(f"\nAI Companion: {response}")

test_chatbot()